In [4]:
import pandas as pd

# Contoh membuka data roster anggota keluarga
df_roster = pd.read_stata('D:/MK4 Rekomendasi Sistem/hh14_all_dta/bk_ar1.dta')
print(df_roster.head())

    hhid14_9  pid14                    ar02b     ar10     ar11    ar10a  \
0  001060000    1.0  1:Head of the household  52:Dead  52:Dead  52:Dead   
1  001060000    2.0                      NaN      NaN      NaN      NaN   
2  001060000    3.0                      NaN      NaN      NaN      NaN   
3  001060000    4.0   3:Children (biological      1.0      2.0  52:Dead   
4  001060000    5.0                      NaN      NaN      NaN      NaN   

     ar11a    ar10b    ar11b                            ar12  ... ar08a  ar09  \
0  52:Dead  52:Dead  52:Dead  96:Not Applicable (>=15 years)  ...  40.0  59.0   
1      NaN      NaN      NaN                             NaN  ...   NaN   NaN   
2      NaN      NaN      NaN                             NaN  ...   NaN   NaN   
3  52:Dead  52:Dead  52:Dead  96:Not Applicable (>=15 years)  ...  14.0  29.0   
4      NaN      NaN      NaN                             NaN  ...   NaN   NaN   

  ar01e ar01g ar01h ar01l ar01m p version  module  
0   3.0   

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# --- 1. FUNGSI LOAD DATA ---
def load_and_fix_ifls(path):
    df = pd.read_stata(path, convert_categoricals=False)
    if 'hhid14_9' in df.columns:
        df = df.rename(columns={'hhid14_9': 'hhid14'})
    df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

base_path = 'D:/MK4 Rekomendasi Sistem/hh14_all_dta/'

df_roster = load_and_fix_ifls(base_path + 'bk_ar1.dta')
df_spend = load_and_fix_ifls(base_path + 'b1_ks1.dta')
df_housing = load_and_fix_ifls(base_path + 'b2_kr.dta')

In [6]:
# --- 2. AGREGASI DATA ---
df_roster['age'] = pd.to_numeric(df_roster['ar08a'], errors='coerce').fillna(0)
df_roster['is_anak'] = df_roster['age'].between(6, 18).astype(int)
df_roster['is_lansia'] = (df_roster['age'] > 60).astype(int)

df_demo_agg = df_roster.groupby('hhid14').agg({
    'is_anak': 'sum',
    'is_lansia': 'sum',
    'age': 'count' 
}).rename(columns={'age': 'jml_anggota'}).reset_index()

spend_col = 'ks02' if 'ks02' in df_spend.columns else 'ks01' 
df_spend[spend_col] = pd.to_numeric(df_spend[spend_col], errors='coerce').fillna(0)
df_spend_agg = df_spend.groupby('hhid14')[spend_col].sum().reset_index()
df_spend_agg.rename(columns={spend_col: 'total_pengeluaran'}, inplace=True)

# ======================================================================
# PROSES PENCANDERAAN KOLOM FISIK RUMAH SECARA FLEKSIBEL (ANTI-KEYERROR)
# ======================================================================
# Mencari kolom yang mengandung teks 'kr01' atau 'kr11' tanpa peduli huruf besar/kecil
floor_candidates = [col for col in df_housing.columns if 'kr01' in col.lower()]
water_candidates = [col for col in df_housing.columns if 'kr11' in col.lower()]

# Menentukan nama kolom final berdasarkan hasil pencarian teks di atas
floor_col = floor_candidates[0] if floor_candidates else df_housing.columns[1]
water_col = water_candidates[0] if water_candidates else df_housing.columns[2]

# Mengonversi nilai kolom menjadi numerik dengan aman
df_housing['jenis_lantai'] = pd.to_numeric(df_housing[floor_col], errors='coerce').fillna(0)
df_housing['sumber_air'] = pd.to_numeric(df_housing[water_col], errors='coerce').fillna(0)

# Ambil kolom esensial perumahan untuk digabungkan
df_housing_clean = df_housing[['hhid14', 'jenis_lantai', 'sumber_air']].drop_duplicates(subset=['hhid14'])

# Penggabungan 3 Buku Utama IFLS
df_final = df_demo_agg.merge(df_spend_agg, on='hhid14', how='left')
df_final = df_final.merge(df_housing_clean, on='hhid14', how='left')
df_final.fillna(0, inplace=True)

,hhid14,is_anak,is_lansia,jml_anggota,total_pengeluaran,jenis_lantai,sumber_air
0,001060000,1,0,11,588000.0,1.0,3.0
1,001060004,1,0,4,188000.0,5.0,3.0
2,001080000,6,0,12,167000.0,2.0,1.0
3,001080003,0,0,7,165500.0,1.0,3.0
4,001220000,4,0,15,742500.0,1.0,3.0
...,...,...,...,...,...,...,...
15916,321280006,0,0,2,457500.0,5.0,1.0
15917,321280009,0,0,2,767000.0,2.0,1.0
15918,321290000,3,1,8,310000.0,1.0,1.0
15919,321291100,2,0,5,759000.0,1.0,1.0


In [7]:
# --- 3. NORMALISASI ---
features = ['is_anak', 'is_lansia', 'jml_anggota', 'total_pengeluaran', 'jenis_lantai', 'sumber_air']
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_final[features])

# Inversi Fitur Sosial-Ekonomi
df_scaled[:, 3] = 1 - df_scaled[:, 3]  # total_pengeluaran
df_scaled[:, 4] = 1 - df_scaled[:, 4]  # jenis_lantai
df_scaled[:, 5] = 1 - df_scaled[:, 5]  # sumber_air

In [8]:
# --- 4. PEMBUATAN MATRIKS PROGRAM ---
program_data = {
    'program': ['PKH', 'BPNT', 'KIP'],
    'is_anak': [0.7, 0.2, 1.0],
    'is_lansia': [0.8, 0.3, 0.0],
    'jml_anggota': [0.6, 0.5, 0.4],
    'total_pengeluaran': [0.6, 1.0, 0.5],
    'jenis_lantai': [0.5, 0.6, 0.3],
    'sumber_air': [0.5, 0.6, 0.3]
}
df_programs = pd.DataFrame(program_data)

In [9]:
# --- 5. EKSPOR ASET BARU ---
joblib.dump(scaler, 'scaler_bansos.pkl')
df_programs.to_csv('df_programs.csv', index=False)
print("Aset baru anti-outlier berhasil diekspor!")

Aset baru anti-outlier berhasil diekspor!


In [10]:
# --- 6. SERIALISASI DAN EKSPOR MODEL (Paling Krusial) ---

# Menyimpan objek scaler agar rentang Min-Max populasi tersimpan permanen
joblib.dump(scaler, 'scaler_bansos.pkl')

# Menyimpan dataframe kriteria program bansos terupdate ke bentuk CSV kecil
df_programs.to_csv('df_programs.csv', index=False)
print("\nAset 'scaler_bansos.pkl' dan 'df_programs.csv' sukses diekspor!")


Aset 'scaler_bansos.pkl' dan 'df_programs.csv' sukses diekspor!
